In [ ]:
import os,sys,subprocess,shutil
OUT='/kaggle/working'
LOG=os.path.join(OUT,'run_log.txt')
def log(m):
    print(m,flush=True)
    with open(LOG,'a') as f: f.write(str(m)+'\n')
log('=== RAW INSIGHTFACE PIPELINE ===')
def find_media(root):
    e={'.mp4','.avi','.mkv','.mov','.wmv','.jpg','.jpeg','.png','.bmp'}
    for dp,dn,fns in os.walk(root):
        for f in fns:
            if os.path.splitext(f)[1].lower() in e: return dp
    return None
DS=find_media('/kaggle/input')
log(f'DS:{DS}')

In [ ]:
# Setup deps + models
subprocess.run(['apt-get','update','-qq'],capture_output=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg'],capture_output=True)
subprocess.run(['pip','install','-q','insightface==0.7.3','onnxruntime-gpu==1.22.0','gfpgan'],capture_output=True)
log('deps ok')

# Fix GFPGAN/basicsr torchvision compat: functional_tensor removed in new torchvision
import torchvision, importlib.util
tv_dir=os.path.dirname(torchvision.__file__)
ft_path=os.path.join(tv_dir,'transforms','functional_tensor.py')
if not os.path.exists(ft_path):
    with open(ft_path,'w') as f:
        f.write('from torchvision.transforms.functional import rgb_to_grayscale\n')
    log('patched torchvision functional_tensor')

MD='/tmp/models'; os.makedirs(MD,exist_ok=True)
ins=os.path.join(MD,'inswapper_128_fp16.onnx')
gfp=os.path.join(MD,'GFPGANv1.4.pth')
if not os.path.exists(ins) or os.path.getsize(ins)<1e6:
    subprocess.run(['wget','-q','-O',ins,'https://huggingface.co/hacksider/deep-live-cam/resolve/main/inswapper_128_fp16.onnx'],check=True)
if not os.path.exists(gfp) or os.path.getsize(gfp)<1e6:
    subprocess.run(['wget','-q','-O',gfp,'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth'],check=True)
log('models ok')

In [ ]:
# Write the worker script (runs in subprocess to avoid numpy import conflicts)
worker='''
import cv2, numpy as np, insightface, sys, os, subprocess
from insightface.app import FaceAnalysis
src_p, vid_p, out_dir, ins_p, gfp_p, use_enh = sys.argv[1:7]
use_enh = use_enh=="1"

app=FaceAnalysis(name="buffalo_l",providers=["CUDAExecutionProvider","CPUExecutionProvider"])
app.prepare(ctx_id=0,det_size=(640,640))
swapper=insightface.model_zoo.get_model(ins_p,providers=["CUDAExecutionProvider","CPUExecutionProvider"])

enhancer=None
if use_enh:
    from gfpgan import GFPGANer
    enhancer=GFPGANer(model_path=gfp_p,upscale=1,arch="clean",channel_multiplier=2,bg_upsampler=None)
    print("ENHANCER_LOADED",flush=True)

# Source face
si=cv2.imread(src_p); sf=app.get(si)
print(f"SRC_FACES={len(sf)}",flush=True)
if not sf: print("NO_SRC_FACE"); sys.exit(1)
src_face=sorted(sf,key=lambda x:(x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]),reverse=True)[0]

# Get video info
cap=cv2.VideoCapture(vid_p)
fps=cap.get(cv2.CAP_PROP_FPS)
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"FPS={fps} SIZE={W}x{H} FRAMES={total}",flush=True)

# Output writer (raw frames -> mp4 via ffmpeg pipe for quality)
tmp_vid=out_dir+"/_noaudio.mp4"
fourcc=cv2.VideoWriter_fourcc(*"mp4v")
vw=cv2.VideoWriter(tmp_vid,fourcc,fps,(W,H))

idx=0; swapped=0
while True:
    ret,frame=cap.read()
    if not ret: break
    faces=app.get(frame)
    if faces:
        for tf in faces:
            frame=swapper.get(frame,tf,src_face,paste_back=True)
        if enhancer is not None:
            try:
                _,_,frame=enhancer.enhance(frame,has_aligned=False,only_center_face=False,paste_back=True)
            except Exception as e:
                pass
        swapped+=1
    vw.write(frame)
    idx+=1
    if idx%200==0: print(f"PROGRESS {idx}/{total} swapped={swapped}",flush=True)
cap.release(); vw.release()
print(f"DONE_FRAMES idx={idx} swapped={swapped}",flush=True)
'''
with open('/tmp/worker.py','w') as f: f.write(worker)
log('worker written')

In [ ]:
# Find input
img_e={'.jpg','.jpeg','.png','.bmp'}; vid_e={'.mp4','.avi','.mkv','.mov','.wmv'}
src=tgt=None
for f in os.listdir(DS):
    ext=os.path.splitext(f)[1].lower(); fp=os.path.join(DS,f)
    if ext in img_e and not src: src=fp
    elif ext in vid_e and not tgt: tgt=fp

# Copy to writable + normalize to mp4 keeping original fps & high quality
W='/tmp/in'; os.makedirs(W,exist_ok=True)
src_w=os.path.join(W,'src'+os.path.splitext(src)[1]); shutil.copy2(src,src_w)
vid_w=os.path.join(W,'in.mp4')
subprocess.run(['ffmpeg','-y','-i',tgt,'-c:v','libx264','-crf','16','-preset','fast','-c:a','aac','-b:a','192k',vid_w],capture_output=True)
log(f'src:{src_w} vid:{os.path.getsize(vid_w)/1e6:.1f}MB')

# Run worker (enhancer ON for sharpness)
log('Running swap pipeline (enhancer ON)...')
p=subprocess.Popen(['python','/tmp/worker.py',src_w,vid_w,OUT,ins,gfp,'1'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    s=line.rstrip()
    if s and any(k in s for k in ['SRC_FACES','FPS=','PROGRESS','DONE_FRAMES','ENHANCER','NO_SRC','Error','error']):
        log('  '+s)
p.wait()
log(f'worker exit={p.returncode}')

In [ ]:
# Mux audio from original video into swapped video
noaudio=OUT+'/_noaudio.mp4'
final=OUT+'/swapped_output.mp4'
if os.path.exists(noaudio) and os.path.getsize(noaudio)>1000:
    r=subprocess.run(['ffmpeg','-y','-i',noaudio,'-i',vid_w,
                      '-c:v','libx264','-crf','16','-preset','fast',
                      '-c:a','aac','-b:a','192k',
                      '-map','0:v:0','-map','1:a:0?',
                      '-movflags','+faststart',final],capture_output=True,text=True)
    if os.path.exists(final) and os.path.getsize(final)>1000:
        os.remove(noaudio)
        log(f'SUCCESS! {os.path.getsize(final)/1e6:.1f}MB')
    else:
        log(f'mux err: {r.stderr[-400:]}')
        shutil.move(noaudio,final)
        log('saved without audio')
else:
    log('FAILED: no video produced')
log('=== DONE ===')